# 05 — Cohort Analizi
`data_clean.csv` üzerinden aylık cohort grupları oluşturulur ve müşteri elde tutma oranları görselleştirilir.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv('data_clean.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'font.family'      : 'DejaVu Serif',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.dpi'       : 150,
})

print(f'Veri yüklendi: {len(df):,} satır')

Veri yüklendi: 387,841 satır


In [2]:
# ── COHORT HESAPLAMA ─────────────────────────────────────────
# CohortMonth : Her müşterinin ilk alışveriş yaptığı ay
# OrderMonth  : İşlemin gerçekleştiği ay
# CohortIndex : İlk alışverişten kaç ay sonra? (0 = ilk ay)

df['CohortMonth'] = df.groupby('CustomerID')['InvoiceDate'] \
                      .transform('min').dt.to_period('M')

df['OrderMonth']  = df['InvoiceDate'].dt.to_period('M')

df['CohortIndex'] = (df['OrderMonth'] - df['CohortMonth']).apply(lambda x: x.n)

# Her cohort × index için benzersiz müşteri sayısı
cohort_data = df.groupby(['CohortMonth','CohortIndex'])['CustomerID'] \
                .nunique().reset_index()
cohort_data.columns = ['CohortMonth','CohortIndex','Customers']

# Pivot tablo
cohort_pivot = cohort_data.pivot(
    index='CohortMonth', columns='CohortIndex', values='Customers'
)

# Elde tutma oranı (her satırı index=0'a böl)
retention = cohort_pivot.divide(cohort_pivot[0], axis=0).round(3)

print('Cohort Pivot (ilk 5 cohort, ilk 6 ay):')
print(cohort_pivot.iloc[:5, :6])
print('\nElde Tutma Oranı (%):')
print((retention.iloc[:5, :6] * 100).round(1))

Cohort Pivot (ilk 5 cohort, ilk 6 ay):
CohortIndex      0      1      2      3      4      5
CohortMonth                                          
2010-12      885.0  324.0  286.0  340.0  321.0  352.0
2011-01      417.0   92.0  111.0   96.0  134.0  120.0
2011-02      380.0   71.0   71.0  108.0  103.0   94.0
2011-03      452.0   68.0  114.0   90.0  101.0   76.0
2011-04      300.0   64.0   61.0   63.0   59.0   68.0

Elde Tutma Oranı (%):
CohortIndex      0     1     2     3     4     5
CohortMonth                                     
2010-12      100.0  36.6  32.3  38.4  36.3  39.8
2011-01      100.0  22.1  26.6  23.0  32.1  28.8
2011-02      100.0  18.7  18.7  28.4  27.1  24.7
2011-03      100.0  15.0  25.2  19.9  22.3  16.8
2011-04      100.0  21.3  20.3  21.0  19.7  22.7


In [3]:
# ── ŞEKİL 12: Cohort Isı Haritası ───────────────────────────

fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(
    retention * 100,
    annot=True, fmt='.0f',
    cmap='YlOrRd_r',
    linewidths=0.4,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Elde Tutma Oranı (%)'},
    vmin=0, vmax=100,
    annot_kws={'size': 8}
)

ax.set_title('Şekil 12. Aylık Cohort Analizi — Müşteri Elde Tutma Oranı (%)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Cohort Ayı (0 = İlk Alışveriş Ayı)', fontsize=11)
ax.set_ylabel('Cohort Grubu (İlk Alışveriş Ayı)', fontsize=11)
ax.set_yticklabels([str(p) for p in retention.index], rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig12_cohort_heatmap.png', bbox_inches='tight')
plt.show()

/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_9771/646680579.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ── ŞEKİL 13: 1. Ay Elde Tutma Oranları ─────────────────────

month1_retention = retention[1].dropna() * 100

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(
    month1_retention.index.astype(str),
    month1_retention.values,
    color='#2980b9', edgecolor='white', linewidth=0.8
)
ax.axhline(
    month1_retention.mean(), color='#e74c3c', linestyle='--',
    linewidth=1.5, label=f'Ortalama: %{month1_retention.mean():.1f}'
)
ax.set_xlabel('Cohort Grubu')
ax.set_ylabel('Elde Tutma Oranı (%)')
ax.set_title('Şekil 13. Cohort Gruplarının 1. Ay Elde Tutma Oranları', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('figures/fig13_cohort_month1.png', bbox_inches='tight')
plt.show()

# Özet
print(f'Ortalama 1. ay tutma : %{month1_retention.mean():.1f}')
print(f'En yüksek            : %{month1_retention.max():.1f}  ({month1_retention.idxmax()})')
print(f'En düşük             : %{month1_retention.min():.1f}  ({month1_retention.idxmin()})')

retention.to_csv('cohort_retention.csv')
print('\n✅ cohort_retention.csv kaydedildi')

Ortalama 1. ay tutma : %20.6
En yüksek            : %36.6  (2010-12)
En düşük             : %11.1  (2011-11)

✅ cohort_retention.csv kaydedildi


/var/folders/84/_5dfw6l50pxgsy8_1qs0_b0w0000gn/T/ipykernel_9771/520251362.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
